# MAPPO Random Wall Proximity Source Curriculum 50x50 Outer / 30x30 Inner


In [ ]:
from pathlib import Path
import os
import sys

# Set these before importing JAX in this kernel.
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.environ.setdefault("XLA_PYTHON_CLIENT_MEM_FRACTION", "0.35")
if "jax" in sys.modules:
    print("Restart the kernel before rerunning training; JAX was already imported.")

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src" / "ant_byte_env").exists():
    raise RuntimeError("Launch this notebook from the cool-antz repo or a subdirectory.")
os.chdir(PROJECT_ROOT)

SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from ant_byte_env import notebook_workflows as workflows

runtime_status = workflows.configure_jax_notebook_runtime()
workflows.assert_notebook_resources_available(runtime_status)
runtime_status


In [ ]:
import importlib

import jax

from ant_byte_env.training.jax_mappo import runner as jax_runner

workflows = importlib.reload(workflows)
jax_runner = importlib.reload(jax_runner)
print(f"JAX device: {jax.devices()[0]}")


## Quick Smoke Run

Run one tiny job before starting the scratch curriculum.

In [ ]:
smoke_metrics = workflows.run_jax_smoke(jax_runner.main)
smoke_metrics


## Curriculum Settings

Edit `experiments/exploration_to_forage_proximity_sources_50x50.json` for durable footprint, reward-scale, margin, or budget changes.

In [ ]:
EXPERIMENT_CONFIG = PROJECT_ROOT / "experiments" / "exploration_to_forage_proximity_sources_50x50_random_walls.json"
experiment = workflows.load_jax_experiment(EXPERIMENT_CONFIG)
experiment_args = dict(experiment.args)

RUN_DIR = PROJECT_ROOT / "runs" / "notebooks" / experiment.name
CHECKPOINT_DIR = RUN_DIR / "checkpoints"
MEDIA_DIR = RUN_DIR / "media"
ROLLOUT_TILE_SIZE = workflows.NOTEBOOK_ROLLOUT_TILE_SIZE
ROLLOUT_POLICY_TEMPERATURE = workflows.notebook_rollout_policy_temperature(experiment.metadata)
WANDB_VIDEO_MAX_FRAMES = int(experiment.metadata["wandb_video_max_frames"])
WANDB_VIDEO_STAGE_NAMES = tuple(experiment.metadata["wandb_preview_stage_names"])
WANDB_VIDEO_ROLLOUT_COUNT = int(experiment.metadata.get("wandb_preview_rollout_count", 1))
CHECKPOINT_VIDEO_INTERVAL_UPDATES = int(experiment.metadata.get("checkpoint_video_interval_updates", 0))
CHECKPOINT_VIDEO_MAX_FRAMES = int(experiment.metadata.get("checkpoint_video_max_frames", WANDB_VIDEO_MAX_FRAMES))
CHECKPOINT_VIDEO_ROLLOUT_COUNT = int(experiment.metadata.get("checkpoint_video_rollout_count", WANDB_VIDEO_ROLLOUT_COUNT))
CHECKPOINT_VIDEO_POLICY_TEMPERATURE = float(experiment.metadata.get("checkpoint_video_policy_temperature", ROLLOUT_POLICY_TEMPERATURE))
CHECKPOINT_VIDEO_WANDB_KEY_PREFIX = experiment.metadata.get(
    "checkpoint_video_wandb_key_prefix",
    "videos/exploration_to_forage/random_walls/checkpoints",
)
STAGE_UPDATE_MULTIPLIER = float(experiment.metadata.get("stage_update_multiplier", 1.0))
SOURCE_CHECKPOINT = None
if experiment_args.get("load_model"):
    SOURCE_CHECKPOINT = workflows.resolve_project_path(PROJECT_ROOT, experiment_args["load_model"])
    if not SOURCE_CHECKPOINT.exists():
        raise FileNotFoundError(f"Run or restore the source checkpoint first: {SOURCE_CHECKPOINT}")
    experiment_args["load_model"] = str(SOURCE_CHECKPOINT)
BEST_CHECKPOINT_PATH = workflows.resolve_project_path(PROJECT_ROOT, experiment_args["save_best_model"])
experiment_args["save_best_model"] = str(BEST_CHECKPOINT_PATH)

SOURCE_COUNTS = tuple(int(count) for count in experiment.metadata["food_source_counts"])
CLUSTER_RADII = tuple(int(radius) for radius in experiment.metadata["food_cluster_radii"])
CURRICULUM_STAGES = workflows.build_food_cluster_curriculum_stages(
    experiment_args,
    source_counts=SOURCE_COUNTS,
    cluster_radii=CLUSTER_RADII,
    visit_reward_schedule=experiment.metadata.get("visit_reward_schedule"),
    view_reward_schedule=experiment.metadata.get("view_reward_schedule"),
    stage_update_multiplier=STAGE_UPDATE_MULTIPLIER,
)
GLOBAL_UPDATE_CAP = max(int(stage["global_update_cap"]) for stage in CURRICULUM_STAGES)
UPDATE_TIMESTEPS = workflows.update_timesteps(
    num_envs=int(experiment_args["num_envs"]),
    num_steps=int(experiment_args["num_steps"]),
)
WANDB_PROJECT = "cool-antz"
WANDB_ENTITY = None
WANDB_GROUP = experiment.name
WANDB_RUN_NAME = WANDB_GROUP
WANDB_MODE = "online"
CRITIC_TAG = f"{experiment_args.get('critic_architecture', 'mlp').replace('_', '-')}-critic"
COMMON_ARGS = workflows.config_common_args(
    experiment_args,
    exclude=workflows.EXPLORATION_TO_FORAGE_ARG_EXCLUDES,
)
{
    "source_checkpoint": SOURCE_CHECKPOINT,
    "best_checkpoint": BEST_CHECKPOINT_PATH,
    "num_ants": experiment_args.get("num_ants"),
    "normal_food_sources": experiment_args.get("food_sources"),
    "lethal_food_sources": experiment_args.get("lethal_food_sources"),
    "lethal_food_distance_range": (
        experiment_args.get("lethal_food_min_distance"),
        experiment_args.get("lethal_food_max_distance"),
    ),
    "random_food_same_distance": experiment_args.get("random_food_same_distance"),
    "random_wall_obstacles": experiment_args.get("random_wall_obstacles"),
    "maze_layout_count": experiment_args.get("maze_layout_count"),
    "random_wall_count_range": (
        experiment_args.get("random_wall_count_min"),
        experiment_args.get("random_wall_count_max"),
    ),
    "random_wall_length_range": (
        experiment_args.get("random_wall_length_min"),
        experiment_args.get("random_wall_length_max"),
    ),
    "random_wall_width": experiment_args.get("random_wall_width"),
    "random_wall_l_turn_probability": experiment_args.get("random_wall_l_turn_probability"),
    "random_wall_center_window_size": experiment_args.get("random_wall_center_window_size"),
    "layout_margin": experiment_args.get("layout_margin"),
    "hub_center_window_size": experiment_args.get("hub_center_window_size"),
    "distance_bonus": experiment_args.get("distance_bonus"),
    "carrying_hub_distance_bonus": experiment_args.get("carrying_hub_distance_bonus"),
    "critic_architecture": experiment_args.get("critic_architecture"),
    "write_bits": experiment_args.get("write_bits"),
    "per_ant_write_channels": experiment_args.get("per_ant_write_channels"),
    "stage_training_profiles": [
        (
            stage["name"],
            stage["food_sources"],
            stage["food_cluster_radius"],
            stage["food_count"],
            stage["global_update_cap"],
            stage["num_steps"],
            stage["gamma"],
        )
        for stage in CURRICULUM_STAGES
    ],
    "total_updates_per_stage": GLOBAL_UPDATE_CAP,
    "update_timesteps": UPDATE_TIMESTEPS,
}


## Train Random Wall Curriculum


In [ ]:
training_result = workflows.run_forage_curriculum(
    stages=CURRICULUM_STAGES,
    checkpoint_dir=CHECKPOINT_DIR,
    common_args=COMMON_ARGS,
    update_timesteps_per_stage=UPDATE_TIMESTEPS,
    global_update_cap=GLOBAL_UPDATE_CAP,
    train_main=jax_runner.main,
    initial_checkpoint=SOURCE_CHECKPOINT,
    wandb_project=WANDB_PROJECT,
    wandb_entity=WANDB_ENTITY,
    wandb_group=WANDB_GROUP,
    wandb_run_name=WANDB_RUN_NAME,
    wandb_mode=WANDB_MODE,
    wandb_tags=[
        "exploration-to-forage",
        "random-walls",
        "near-nest-walls",
        "lethal-cookies",
        "single-lethal-cookie",
        "sparse-cookies",
        "2-ants",
        CRITIC_TAG,
        "50x50",
        "30x30-inner",
    ],
    wandb_notes=experiment.metadata["notes"],
    wandb_artifact_paths=[EXPERIMENT_CONFIG],
    wandb_artifact_prefix="exploration-to-forage-random-walls",
    checkpoint_name_prefix="jax_mappo_exploration_to_forage_random_walls",
    wandb_video_key_prefix="videos/exploration_to_forage/random_walls",
    wandb_video_max_frames=WANDB_VIDEO_MAX_FRAMES,
    wandb_video_stage_names=WANDB_VIDEO_STAGE_NAMES,
    wandb_video_policy_temperature=ROLLOUT_POLICY_TEMPERATURE,
    wandb_video_rollout_count=WANDB_VIDEO_ROLLOUT_COUNT,
    checkpoint_video_interval_updates=CHECKPOINT_VIDEO_INTERVAL_UPDATES,
    checkpoint_video_max_frames=CHECKPOINT_VIDEO_MAX_FRAMES,
    checkpoint_video_policy_temperature=CHECKPOINT_VIDEO_POLICY_TEMPERATURE,
    checkpoint_video_rollout_count=CHECKPOINT_VIDEO_ROLLOUT_COUNT,
    checkpoint_video_wandb_key_prefix=CHECKPOINT_VIDEO_WANDB_KEY_PREFIX,
)
FINAL_CHECKPOINT_PATH = training_result["final_checkpoint_path"]
ROLLOUT_CHECKPOINT_PATH = FINAL_CHECKPOINT_PATH
training_result


## Optional Local Render


In [ ]:
# rollout_result = workflows.render_jax_checkpoint_rollout(
#     run_dir=RUN_DIR,
#     checkpoint_path=BEST_CHECKPOINT_PATH,
#     media_dir=MEDIA_DIR,
#     rollout_filename="jax_mappo_exploration_to_forage_random_walls_rollout.mp4",
#     title="JAX MAPPO random wall curriculum rollout",
#     description="Sampled rollout from the random-wall lethal-cookie checkpoint.",
#     metadata={
#         "experiment_config": str(EXPERIMENT_CONFIG),
#         "source_checkpoint": str(SOURCE_CHECKPOINT),
#         "best_checkpoint": str(BEST_CHECKPOINT_PATH),
#         "num_ants": experiment_args.get("num_ants"),
#         "normal_food_sources": experiment_args.get("food_sources"),
#         "lethal_food_sources": experiment_args.get("lethal_food_sources"),
#         "lethal_food_min_distance": experiment_args.get("lethal_food_min_distance"),
#         "lethal_food_max_distance": experiment_args.get("lethal_food_max_distance"),
#         "random_wall_obstacles": experiment_args.get("random_wall_obstacles"),
#         "maze_layout_count": experiment_args.get("maze_layout_count"),
#         "random_wall_count_min": experiment_args.get("random_wall_count_min"),
#         "random_wall_count_max": experiment_args.get("random_wall_count_max"),
#         "random_wall_length_min": experiment_args.get("random_wall_length_min"),
#         "random_wall_length_max": experiment_args.get("random_wall_length_max"),
#         "random_wall_width": experiment_args.get("random_wall_width"),
#         "random_wall_l_turn_probability": experiment_args.get("random_wall_l_turn_probability"),
#         "random_wall_center_window_size": experiment_args.get("random_wall_center_window_size"),
#         "food_source_counts": [stage["food_sources"] for stage in CURRICULUM_STAGES],
#         "food_cluster_radii": [stage["food_cluster_radius"] for stage in CURRICULUM_STAGES],
#     },
#     tile_size=ROLLOUT_TILE_SIZE,
#     policy_temperature=ROLLOUT_POLICY_TEMPERATURE,
#     reuse_existing=False,
#     wandb_project=WANDB_PROJECT,
#     wandb_entity=WANDB_ENTITY,
#     wandb_group=WANDB_GROUP,
#     wandb_run_name=f"{WANDB_GROUP}_rollout",
#     wandb_mode="disabled",
#     wandb_video_key=None,
# )
# rollout_result
